# equiml — Pre-processing Fairness Tutorial

This notebook demonstrates the pre-processing fairness methods available in `equiml` on two datasets:

1. **Adult Income** (UCI) — a classic fairness benchmark
2. **ACS / folktables** — a more recent and realistic benchmark (Ding et al. 2021)

For each dataset we will:
- Train a baseline model and audit its fairness
- Apply each pre-processing method
- Retrain and re-audit
- Compare fairness / accuracy tradeoffs

**Pre-processing methods covered:**
- `Reweighting` (Kamiran & Calders, 2012)
- `WassersteinRepair` quantile (Feldman et al., 2015)
- `WassersteinRepair` OT — Random Repair (Gordaliza et al., 2019)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from equiml import audit
from equiml.mitigation import Reweighting, WassersteinRepair

## Helper functions

In [ ]:
def train_and_audit(X_train, y_train, X_test, y_test, sensitive_test,
                    sample_weight=None, label="Baseline"):
    """Train a logistic regression and audit its fairness."""
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train, sample_weight=sample_weight)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)

    result_labels = audit(y_test, y_pred, sensitive_test,
                          metrics=["demographic_parity", "equalized_odds",
                                   "equal_opportunity"])
    result_proba = audit(y_test, y_proba, sensitive_test,
                         metrics=["auc_parity"])

    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(f"  Accuracy              : {acc:.4f}")
    print(f"  Demographic Parity    : {result_labels.metrics['demographic_parity'].value:.4f}")
    print(f"  Equal Opportunity     : {result_labels.metrics['equal_opportunity'].value:.4f}")
    print(f"  Equalized Odds        : {result_labels.metrics['equalized_odds'].value:.4f}")
    print(f"  AUC Parity            : {result_proba.metrics['auc_parity'].value:.4f}")

    return {
        "label": label,
        "accuracy": acc,
        "demographic_parity": result_labels.metrics["demographic_parity"].value,
        "equal_opportunity": result_labels.metrics["equal_opportunity"].value,
        "equalized_odds": result_labels.metrics["equalized_odds"].value,
        "auc_parity": result_proba.metrics["auc_parity"].value,
    }

---
# Part 1 — Adult Income Dataset

## 1.1 Loading the data

In [ ]:
df = pd.read_csv("adult_norm.csv")

# Sensitive attribute and label
sensitive_col = "gender:is_Male"
label_col = "income:is_>50K"

# Numerical features — repair will be applied only to these
numerical_cols = ["age", "educational-num", "capital-gain",
                  "capital-loss", "hours-per-week"]
numerical_idx = [df.drop(columns=[sensitive_col, label_col]).columns
                   .get_loc(c) for c in numerical_cols]

X = df.drop(columns=[sensitive_col, label_col]).values
y = df[label_col].values
sensitive = df[sensitive_col].values

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Positive rate overall   : {y.mean():.3f}")
print(f"Positive rate (male)    : {y[sensitive == 1].mean():.3f}")
print(f"Positive rate (female)  : {y[sensitive == 0].mean():.3f}")

## 1.2 Train/test split

In [ ]:
X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X, y, sensitive, test_size=0.3, random_state=42, stratify=y
)

results_adult = []

## 1.3 Baseline

In [ ]:
res = train_and_audit(X_train, y_train, X_test, y_test, s_test,
                      label="Baseline (no mitigation)")
results_adult.append(res)

## 1.4 Reweighting (Kamiran & Calders 2012)

In [ ]:
rw = Reweighting()
result_rw = rw.fit_transform(X_train, y_train, s_train)

res = train_and_audit(result_rw.X, result_rw.y, X_test, y_test, s_test,
                      sample_weight=result_rw.weights,
                      label="Reweighting")
results_adult.append(res)

## 1.5 WassersteinRepair — quantile (Feldman et al. 2015)

In [ ]:
for lambda_ in [0.5, 1.0]:
    # Repair numerical features only
    repair = WassersteinRepair(
        method="quantile",
        repair_type="geometric",
        lambda_=lambda_
    )
    result_rep = repair.fit_transform(
        X_train[:, numerical_idx], y_train, s_train
    )

    X_train_rep = X_train.copy()
    X_train_rep[:, numerical_idx] = result_rep.X

    # Apply same transform to test set
    result_test = repair.transform(
        X_test[:, numerical_idx], y_test, s_test
    )
    X_test_rep = X_test.copy()
    X_test_rep[:, numerical_idx] = result_test.X

    res = train_and_audit(
        X_train_rep, y_train, X_test_rep, y_test, s_test,
        label=f"WassersteinRepair quantile (λ={lambda_})"
    )
    results_adult.append(res)

## 1.6 WassersteinRepair — OT Random Repair (Gordaliza et al. 2019)

In [ ]:
for lambda_ in [0.5, 1.0]:
    repair_ot = WassersteinRepair(
        method="ot",
        repair_type="random",
        lambda_=lambda_,
        random_state=42
    )
    result_ot = repair_ot.fit_transform(
        X_train[:, numerical_idx], y_train, s_train
    )

    # Reconstruct full feature matrix
    # Note: OT repair may change dataset size
    n_rep = len(result_ot.y)
    X_train_ot = np.zeros((n_rep, X_train.shape[1]))

    # Rebuild sensitive array for repaired dataset
    n0 = (s_train == 0).sum()
    # Approximate: keep track of group sizes
    # (stored in extra by the mitigator)
    n0_rep = result_ot.extra.get("n_repaired", n_rep) // 2
    s_train_ot = np.array([0] * n0_rep + [1] * (n_rep - n0_rep))

    # Fill numerical features with repaired values
    X_train_ot[:, numerical_idx] = result_ot.X if result_ot.X.ndim > 1 else result_ot.X.reshape(-1, 1)

    res = train_and_audit(
        X_train_ot, result_ot.y, X_test, y_test, s_test,
        sample_weight=result_ot.weights,
        label=f"WassersteinRepair OT (λ={lambda_})"
    )
    results_adult.append(res)

## 1.7 Comparison

In [ ]:
df_results = pd.DataFrame(results_adult)
df_results = df_results.set_index("label")
df_results = df_results.round(4)
print("\nAdult Income — Fairness / Accuracy comparison")
print(df_results.to_string())

---
# Part 2 — ACS / folktables (Ding et al. 2021)

folktables provides more recent data from the American Community Survey (ACS).
We use the `ACSIncome` task, which predicts whether an individual earns more than $50,000/year.

**Reference:** Ding, F., Hardt, M., Miller, J., & Schmidt, L. (2021). Retiring Adult: New Datasets for Fair Machine Learning. NeurIPS 2021. https://arxiv.org/abs/2108.04884

In [ ]:
try:
    from folktables import ACSDataSource, ACSIncome
except ImportError:
    raise ImportError(
        "folktables is required for this section. "
        "Install it with: pip install folktables"
    )

## 2.1 Loading ACS data

In [ ]:
# Download 2018 data for California
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
acs_data = data_source.get_data(states=["CA"], download=True)

X_acs, y_acs, group_acs = ACSIncome.df_to_numpy(acs_data)

# Sensitive attribute: SEX (1=male, 2=female) → binarize
sensitive_acs = (group_acs == 1).astype(int)  # 1=male, 0=female

print(f"ACS dataset: {X_acs.shape[0]} samples, {X_acs.shape[1]} features")
print(f"Positive rate overall  : {y_acs.mean():.3f}")
print(f"Positive rate (male)   : {y_acs[sensitive_acs == 1].mean():.3f}")
print(f"Positive rate (female) : {y_acs[sensitive_acs == 0].mean():.3f}")

## 2.2 Train/test split and subsampling

In [ ]:
# Normalize features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_acs_scaled = scaler.fit_transform(X_acs.astype(float))

X_tr, X_te, y_tr, y_te, s_tr, s_te = train_test_split(
    X_acs_scaled, y_acs, sensitive_acs,
    test_size=0.3, random_state=42, stratify=y_acs
)

# All features are numerical for ACS
numerical_idx_acs = list(range(X_acs_scaled.shape[1]))
results_acs = []

# --- Subsampling for repair methods ---
# OT repair is O(n²) in memory — not feasible on the full dataset.
# We subsample the training set for repair methods only.
# The model is still evaluated on the full test set.
REPAIR_SAMPLE_SIZE = 5000
np.random.seed(42)
idx_sample = np.random.choice(len(X_tr), size=REPAIR_SAMPLE_SIZE, replace=False)
X_tr_small = X_tr[idx_sample]
y_tr_small = y_tr[idx_sample]
s_tr_small = s_tr[idx_sample]

print(f"Full train size    : {len(X_tr)}")
print(f"Repair sample size : {len(X_tr_small)}")
print(f"Test size          : {len(X_te)}")

## 2.3 Baseline

In [ ]:
res = train_and_audit(X_tr, y_tr, X_te, y_te, s_te,
                      label="Baseline (no mitigation)")
results_acs.append(res)

## 2.4 Reweighting

In [ ]:
rw_acs = Reweighting()
result_rw_acs = rw_acs.fit_transform(X_tr_small, y_tr_small, s_tr_small)

res = train_and_audit(result_rw_acs.X, result_rw_acs.y,
                      X_te, y_te, s_te,
                      sample_weight=result_rw_acs.weights,
                      label="Reweighting")
results_acs.append(res)

## 2.5 WassersteinRepair — quantile

In [ ]:
for lambda_ in [0.5, 1.0]:
    repair_acs = WassersteinRepair(
        method="quantile",
        repair_type="geometric",
        lambda_=lambda_
    )
    result_rep_acs = repair_acs.fit_transform(X_tr_small, y_tr_small, s_tr_small)
    result_test_acs = repair_acs.transform(X_te, y_te, s_te)

    res = train_and_audit(
        result_rep_acs.X, y_tr_small,
        result_test_acs.X, y_te, s_te,
        label=f"WassersteinRepair quantile (λ={lambda_})"
    )
    results_acs.append(res)

## 2.6 WassersteinRepair — OT Random Repair

⚠️ **Scalability note**: The OT repair solves a linear program of size 
$n_0 \times n_1$, which is $O(n^2)$ in memory. On large datasets like ACS, we apply it on a subsample of the training set. As a consequence, the repaired model may not generalize as well as on the full dataset. This is a known limitation of the exact OT approach. For large-scale applications, approximate OT methods (e.g. Sinkhorn regularization) would be needed.

In [ ]:
for lambda_ in [0.5, 1.0]:
    repair_ot_acs = WassersteinRepair(
        method="ot",
        repair_type="random",
        lambda_=lambda_,
        random_state=42
    )
    result_ot_acs = repair_ot_acs.fit_transform(X_tr_small, y_tr_small, s_tr_small)

    res = train_and_audit(
        result_ot_acs.X, result_ot_acs.y,
        X_te, y_te, s_te,
        sample_weight=result_ot_acs.weights,
        label=f"WassersteinRepair OT (λ={lambda_})"
    )
    results_acs.append(res)

## 2.7 Comparison

In [ ]:
df_results_acs = pd.DataFrame(results_acs)
df_results_acs = df_results_acs.set_index("label")
df_results_acs = df_results_acs.round(4)
print("\nACS / folktables — Fairness / Accuracy comparison")
print(df_results_acs.to_string())